# HW Simulation Pipeline (Assuming Interfaces Are Ready)

This notebook validates the end-to-end hardware simulation flow:
1. Create NAND weight file via `NandFileSystem`
2. Build a manual kernel: `NandMmap -> SramPrefetch -> MatMul`
3. Initialize xPU and load runtime tables
4. Run simulation with `SimSession.scheduler.run()`

Assumptions:
- Runtime and hardware interfaces are already implemented.
- Logical addresses in macro ops are page indices.


In [1]:
import math

from Desim import SimSession

from nandmachine.config.config import NandConfig
from nandmachine.commands.macro import MatMul, NandMmap, SramPrefetch
from nandmachine.simulator.hardware.xpu import xPU
from nandmachine.simulator.runtime.manager import NandFileSystem
from nandmachine.simulator.runtime.tables import DeviceType, NandFileMeta, Permission


In [2]:
WEIGHT_M = 8192
WEIGHT_N = 8192
WEIGHT_BITS = 16
BATCH = 512

nand_config = NandConfig(
    num_channels=8 * 4,
    num_plane=64,
    num_block=512,
    num_pages=256,
    tRead=4000,
    tWrite=40000,
    tErase=400000,
    page_size=4,
)

PAGE_BYTES = nand_config.page_size_bytes
weight_bytes = WEIGHT_M * WEIGHT_N * (WEIGHT_BITS // 8)
weight_pages = math.ceil(weight_bytes / PAGE_BYTES)

print("weight_bytes:", weight_bytes)
print("weight_pages:", weight_pages)


weight_bytes: 134217728
weight_pages: 32768


In [3]:
nfs = NandFileSystem(nand_config)

weight_meta = NandFileMeta(
    file_name="weight_8192x8192_fp16.bin",
    num_pages=weight_pages,
    file_size=weight_bytes,
    permission=Permission.READ,
    type="weight",
)

weight_file_id = nfs.create_static_file(weight_meta)
print("weight_file_id:", weight_file_id)


weight_file_id: 0


In [4]:
# Logical addresses are page indices
weight_logic_base = 1 << 20
sram_logic_base = weight_logic_base + weight_pages + 1024

kernel_commands = [
    NandMmap(file_id=weight_file_id, pre_alloc_logic_addr=weight_logic_base),
    SramPrefetch(
        prefetch_addr=weight_logic_base,
        num_pages=weight_pages,
        pre_alloc_logic_addr=sram_logic_base,
    ),
    MatMul(
        dim=(BATCH, WEIGHT_M, WEIGHT_N),
        addr=sram_logic_base,
        weight_bits=WEIGHT_BITS,
    ),
]

print("kernel_len:", len(kernel_commands))


kernel_len: 3


In [5]:
SimSession.reset()
SimSession.init()

xpu = xPU(nand_config)
xpu.runtime_manager.load_nand_file_system(
    nfs.nand_file_table,
    nfs.nand_free_table,
)
xpu.load_command(kernel_commands)

print("xPU initialized and command queue loaded.")


xPU initialized and command queue loaded.


In [6]:
SimSession.scheduler.run()
print("sim_cycle:", SimSession.sim_time.cycle)

runtime_manager = xpu.runtime_manager
src_map = runtime_manager.page_table.translate(weight_logic_base)
dst_map = runtime_manager.page_table.translate(sram_logic_base)

assert src_map is not None and src_map[0] == DeviceType.NAND
assert dst_map is not None and dst_map[0] == DeviceType.SRAM

print("Simulation completed.")
print("weight_file_id:", weight_file_id)
print("weight_pages:", weight_pages)
print("sim_cycle:", SimSession.sim_time.cycle)


sim_cycle: 0


AssertionError: 